# IRIS — reproducibility overview

This notebook is the entry point. It checks that the environment and every
input dataset resolve, then summarises the perturbation atlas the models are
trained on.

**Kernel:** use the `scvi-env-new` environment
(`/lab/li_lab/Nicholas_keep/miniconda3/envs/scvi-env-new/bin/python`), which
matches the versions used for the published figures.

Notebook map:

| notebook | figure | what it does |
|---|---|---|
| `01_fig1_response_genes.ipynb` | Fig. 1c, Supp. 1 | response-gene baseline vs IRIS, in-sample |
| `02_fig2_generalization.ipynb` | Fig. 2c–f | cross-screen / cross-cell-type / ablation |
| `03_fig3_in_vivo_lineages.ipynb` | Fig. 3 | signaling histories in the mouse embryo |
| `04_fig4_mesenchyme.ipynb` | Fig. 4 | organ-specific mesenchyme + protocol design |

Each notebook mirrors a script under `figures/`; the scripts are the
authoritative, non-interactive path.

In [ ]:
import sys, os
from pathlib import Path

# Make the package importable no matter where Jupyter was started.
ROOT = Path.cwd()
while not (ROOT / "src" / "iris_repro").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

import numpy as np
import pandas as pd
from iris_repro import config, data, metrics, plotting, provenance

plt = plotting.set_style()
provenance.check_environment()
print("package root:", ROOT)
print("outputs ->", config.load_config()["roots"]["outputs"])

## 1. Configuration\n\nAll paths and hyperparameters come from `config/*.yaml`.

In [ ]:
cfg = config.load_config()
print("roots:")
for k, v in cfg["roots"].items():
    print(f"  {k:8s} {v}")

print("\narchitectures (hidden / latent, 1 layer):")
for sig in config.signals(include_shh=True):
    a = config.architecture(sig)
    print(f"  {sig:5s} hidden={a['n_hidden']:5d}  latent={a['n_latent']}")

print("\ntraining:", config.training_params())

## 2. Do all inputs resolve?\n\nA red line here means a path in `config/paths.yaml` needs updating.

In [ ]:
rows = []
for key in cfg["data"]:
    try:
        p = config.data_path(key)
        rows.append({"dataset": key, "status": "OK",
                     "GB": round(p.stat().st_size / 1e9, 2), "path": str(p)})
    except Exception as e:
        rows.append({"dataset": key, "status": "MISSING", "GB": None,
                     "path": str(e)[:70]})
pd.DataFrame(rows)

## 3. The perturbation atlas\n\nSix collection batches spanning two species and three lineages.

In [ ]:
batches = pd.DataFrame(config.batch_table()).T
batches.index.name = "batch"
batches[["name", "species", "lineage", "n_cells", "description"]]

In [ ]:
# Loading the full reference takes ~1 min and a few GB of RAM.
adata = data.load_screens()
print(adata)
data.describe(adata)

### Stimulated fraction per pathway per screen\n\nThis is the class balance every metric is measured against.

In [ ]:
summary = data.describe(adata).set_index("screen")
frac = summary[[c for c in summary.columns if c.endswith("_stim_frac")]]
frac.columns = [c.replace("_stim_frac", "") for c in frac.columns]

fig, ax = plt.subplots(figsize=(4, 2.2))
im = ax.imshow(frac.values, cmap="RdBu_r", vmin=0, vmax=1, aspect="auto")
ax.set_xticks(range(len(frac.columns)))
ax.set_xticklabels([config.display_name(c) for c in frac.columns])
ax.set_yticks(range(len(frac.index)))
ax.set_yticklabels(frac.index)
for i in range(frac.shape[0]):
    for j in range(frac.shape[1]):
        ax.text(j, i, f"{frac.values[i, j]:.2f}", ha="center", va="center", fontsize=5)
ax.set_title("Fraction of cells stimulated")
fig.colorbar(im, ax=ax, shrink=0.8)
plt.show()